In [23]:
import numpy as np

from weather.config import (
    Experiment,
    WeatherFixedParams,
    WeatherGridParams,
    MLPFixedParams,
    MLPGridParams,
    FitFixedParams,
    FitGridParams,
)

from weather.search import Search

from mlp.utils import (
    plot_loss,
    regression_report,
    classification_report_binary,
    plot_roc_auc,
    plot_accuracy,
    accuracy_within_tolerance,
)


In [24]:
SEED = 42
np.random.seed(SEED)


In [25]:
# =========================================================
# 2) WIND (>=6 m/s) BINARY CLASSIFICATION
# =========================================================
exp_wind_encoding = Experiment(
    name="wind6_binary_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="wind_speed",
        target_mode="binary",
        target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation="flatten",
        window_size=3,
        normalization="standardize",
        input_variables=[
            ("wind_speed",),
            # ("wind_speed", "wind_direction"),
            # ("wind_speed", "wind_direction", "pressure"),
            # ("wind_speed", "wind_direction", "pressure", "humidity"),
            # ("wind_speed", "wind_direction", "pressure", "humidity", "temperature"),
        ],
        aggregations={
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        cities=[
            # ("Vancouver",),
            ("Eilat",),
            # ("Vancouver", "Seattle", "Portland"),
            # ("Beersheba", "Tel Aviv District", "Eilat", "Haifa", "Nahariyya", "Jerusalem")
        ]
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="binary",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers = [
            (128, 64),
        ],
        loss="binary_cross_entropy",
        activation="gelu",
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=100,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)


In [26]:
search = Search()

results2 = search.run(exp_wind_encoding)


Starting experiment: wind6_binary_encoding

Building dataset
  → TRAIN split


Eilat | windows: 100%|██████████| 1518/1518 [00:02<00:00, 608.07it/s]


  → TEST split


Eilat | windows: 100%|██████████| 361/361 [00:00<00:00, 611.92it/s]



Configuration run 1/1:
WEATHER (variable):
  - input_variables: ('wind_speed',)
  - cities: ('Eilat',)
MLP (variable):
  - hidden_layers: (128, 64)

Training model


Training:  26%|██▌       | 102/400 [00:12<00:35,  8.37it/s, acc=0.7862, loss=0.4826, lr=0.00358748]


Early stopping at epoch 103, best val_loss=0.422594, train_acc=0.7862, val_acc=0.8224 after 100 epochs without improvement.
Training finished in 12.20 seconds

Experiment finished | total runs = 1



In [27]:
from IPython.core.display import HTML

for run in results2:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    if model.task != "binary":
        raise ValueError("This analysis is only for binary classification tasks.")

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    metrics = classification_report_binary(
        y_true=y_test,
        y_score=y_proba,
        threshold=0.5,
    )

    auc_val = metrics["auc"]

    # if auc_val <= 0.7:
    #     continue

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================

    if auc_val >= 0.72:
        auc_color = "#2e7d32"   # dark green
    elif auc_val >= 0.70:
        auc_color = "#558b2f"   # olive green
    elif auc_val >= 0.65:
        auc_color = "#f9a825"   # amber
    elif auc_val >= 0.6:
        auc_color = "#ef6c00"   # orange
    else:
        auc_color = "#c62828"   # red

    print("=== TEST METRICS (BINARY CLASSIFICATION) ===")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"Auc      : {metrics['auc']:.4f}")
    display(HTML(
        f"""
        <div style="
            font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
            font-size: 13px;
            color: {auc_color};
            padding-left: 12px;
            margin: 4px 0;
        ">
            <b>AUC</b>: {auc_val:.4f}
        </div>
        """
    ))

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - input_variables: ('wind_speed',)
    - cities: ('Eilat',)
  MLP:
    - hidden_layers: (128, 64)
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.7988
Precision: 0.8897
Recall   : 0.8705
Auc      : 0.7214
